# YOLOv8s Hand Detector — Train on HaGRID (Colab end-to-end)

**Capstone CV 2025.2** — Stage 1 detector training.

Notebook full pipeline: download HaGRIDv2 → convert annotations → train YOLOv8s → log W&B → save Drive. Chỉ cần **Run All**.

## Yêu cầu
1. **Bật T4 GPU**: Runtime → Change runtime type → T4 GPU.
2. **W&B API key**: https://wandb.ai/authorize (login khi cell hỏi).
3. **Google Drive** mount để lưu checkpoint.
4. **Đừng đóng tab Colab giữa chừng** (idle > 90 phút sẽ disconnect).

## Lưu ý disk + time

- Colab free disk = **107 GB**. Notebook mặc định chỉ download **1 gesture** (`call.zip` ~37 GB) để fit thoải mái.
- Strategy disk: download zip → extract → **xóa zip ngay** → giữ chỗ cho training.
- Tổng thời gian ước tính:
  - Download + extract: ~30 phút
  - Convert annotations: ~5 phút
  - Train 100 epochs YOLOv8s: ~2.5 giờ trên T4
- Nếu muốn nhiều gestures hơn → cell `Configuration` có thể thêm vào list, nhưng coi chừng đầy disk.

## Trade-off

Train YOLOv8s trên 1 gesture (~30k ảnh) so với pretrained YOLOv10n_hands của HaGRID team (1M ảnh):
- mAP@0.5 expected: **0.75–0.82** (thua pretrained 0.879 — diện ảnh nhỏ hơn 30x)
- Ưu điểm: bạn tự train được → narrative "fine-tuned on HaGRID" trong báo cáo
- Nếu chỉ ưu tiên mAP cao → dùng `colab_download_hagrid_pretrained.ipynb` (mAP 87.9% ngay)

## 0 · Setup + GPU + Drive

In [ ]:
import torch, os, shutil
print(f'PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    !nvidia-smi -L

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/signlang'
os.makedirs(SAVE_DIR, exist_ok=True)

!pip install -q ultralytics wandb tqdm
print('Setup OK.')

## 0.5 · W&B login (làm sớm để Run All không bị treo)

Cell này hỏi API key ngay từ đầu. Sau khi xong, mọi cell tiếp theo (download 30 phút + train 2.5 giờ) chạy không cần bạn ngồi canh.

In [ ]:
import wandb
wandb.login()   # paste W&B API key (lấy ở https://wandb.ai/authorize)
print('✓ W&B login OK. Bạn có thể đi pha cà phê trong khi notebook chạy.')

## 1 · Configuration — gestures + hyperparams

**Mặc định**: 1 gesture (`call`, 37 GB, ~30k ảnh). An toàn cho disk Colab.

Thêm gestures nếu bạn có Colab Pro / High-RAM (more disk):

```python
GESTURES_TO_DOWNLOAD = ['call', 'palm']   # 2 gestures, ~80 GB tổng — TIGHT, có thể oom disk
```

Danh sách gestures hợp lệ + size (GB): `call`(37), `dislike`(41), `fist`(42), `four`(43), `like`(không có size công bố), `mute`(không), `ok`(không), `palm`(43), `peace`(41), `peace_inverted`(40), `rock`(42), `stop`(42), `stop_inverted`(không), `three`(không), `three2`(không), `two_up`(không), `two_up_inverted`(41).

In [ ]:
# === Dataset config ===
GESTURES_TO_DOWNLOAD = ['call']        # mặc định 1 gesture; thêm nếu disk cho phép
DATASET_ROOT = '/content/hagrid'
ANNOTATIONS_URL = 'https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid_v2/annotations_with_landmarks/annotations.zip'

GESTURE_URL_TEMPLATE = 'https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid/hagrid_dataset_new_554800/hagrid_dataset/{gesture}.zip'
# Một số gesture v2-only nằm path khác; nếu URL trên 404, fallback URL:
GESTURE_URL_FALLBACK = 'https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid_v2/hagrid_v2_zip/{gesture}.zip'

# === Training config ===
config = {
    'model_variant': 'yolov8s.pt',
    'epochs': 100,
    'batch': 16,
    'imgsz': 640,
    'patience': 15,
    'optimizer': 'SGD',
    'lr0': 0.01,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'cos_lr': True,
    'mosaic': 1.0,
    'mixup': 0.1,
    'amp': True,
    'workers': 4,
    'seed': 11711,
    'val_split': 0.10,
    'project_name': 'signlang-detector',
    'run_name': 'yolov8s-hagrid-' + '-'.join(GESTURES_TO_DOWNLOAD),
    'gestures': GESTURES_TO_DOWNLOAD,
}
print(config)

## 2 · Download HaGRID annotations (~few hundred MB)

In [ ]:
import urllib.request, zipfile
from tqdm.auto import tqdm

def download_with_progress(url, dest):
    if os.path.exists(dest):
        print(f'  Skipped (exists): {dest} ({os.path.getsize(dest)/1024**3:.2f} GB)')
        return
    req = urllib.request.Request(url)
    with urllib.request.urlopen(req) as r, open(dest, 'wb') as f:
        total = int(r.headers.get('Content-Length') or 0)
        with tqdm(total=total, unit='B', unit_scale=True, unit_divisor=1024, desc=os.path.basename(dest)) as bar:
            while True:
                chunk = r.read(8192 * 16)
                if not chunk:
                    break
                f.write(chunk); bar.update(len(chunk))

os.makedirs(DATASET_ROOT, exist_ok=True)
ann_zip = f'{DATASET_ROOT}/annotations.zip'
ann_dir = f'{DATASET_ROOT}/annotations'

if not os.path.isdir(ann_dir):
    download_with_progress(ANNOTATIONS_URL, ann_zip)
    print(f'Extracting annotations…')
    with zipfile.ZipFile(ann_zip) as z:
        z.extractall(ann_dir)
    os.remove(ann_zip)
    print(f'  ✓ Extracted to {ann_dir}, removed zip')

# Show what's inside
for root, dirs, files in os.walk(ann_dir):
    for d in dirs[:3]:
        print(f'  {root}/{d}/')
    for f in files[:5]:
        print(f'  {root}/{f}')
    if len(dirs) > 3 or len(files) > 5:
        print(f'  …(more in {root})')
    break
!df -h /content | tail -1

## 3 · Download gesture images — 1 cái 1 cái, xóa zip sau khi extract

Cell này có thể mất 20–40 phút mỗi gesture (tùy bandwidth Sber Cloud).

In [ ]:
img_root = f'{DATASET_ROOT}/images'
os.makedirs(img_root, exist_ok=True)

for gesture in GESTURES_TO_DOWNLOAD:
    gesture_dir = f'{img_root}/{gesture}'
    if os.path.isdir(gesture_dir) and len(os.listdir(gesture_dir)) > 100:
        print(f'  ✓ Already extracted: {gesture}')
        continue

    zip_path = f'{DATASET_ROOT}/{gesture}.zip'
    url = GESTURE_URL_TEMPLATE.format(gesture=gesture)
    try:
        download_with_progress(url, zip_path)
    except Exception as e:
        print(f'  Primary URL failed: {e}; trying fallback…')
        url = GESTURE_URL_FALLBACK.format(gesture=gesture)
        download_with_progress(url, zip_path)

    print(f'Extracting {gesture}.zip…')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(img_root)
    os.remove(zip_path)
    print(f'  ✓ Extracted, deleted zip. Disk:')
    !df -h /content | tail -1

# Sanity check
print('\nGesture folders:')
for gesture in GESTURES_TO_DOWNLOAD:
    g_dir = f'{img_root}/{gesture}'
    if os.path.isdir(g_dir):
        n = len(os.listdir(g_dir))
        print(f'  {gesture}: {n} files')
    else:
        print(f'  ⚠️  {gesture}: NOT FOUND')

## 4 · Convert HaGRID JSON → YOLO labels

HaGRID format: `bboxes = [[x_topleft, y_topleft, w, h], …]` (normalized).
YOLO format: `class_id cx cy w h` (normalized). Treat all gestures as class 0 (`hand`).

In [ ]:
import json, glob, random
from pathlib import Path

yolo_root = f'{DATASET_ROOT}/yolo'
for split in ['train', 'val']:
    os.makedirs(f'{yolo_root}/images/{split}', exist_ok=True)
    os.makedirs(f'{yolo_root}/labels/{split}', exist_ok=True)

# Find annotation files for our gestures
ann_files = []
for split_dir in ['train', 'train_val', 'val', 'test']:
    for gesture in GESTURES_TO_DOWNLOAD:
        candidates = (
            glob.glob(f'{ann_dir}/**/{split_dir}/{gesture}.json', recursive=True)
            + glob.glob(f'{ann_dir}/{split_dir}/{gesture}.json')
        )
        for c in candidates:
            if c not in [a[0] for a in ann_files]:
                ann_files.append((c, gesture))

if not ann_files:
    # Fallback: any .json in annotations matching gesture name
    for gesture in GESTURES_TO_DOWNLOAD:
        for c in glob.glob(f'{ann_dir}/**/{gesture}.json', recursive=True):
            ann_files.append((c, gesture))

print(f'Found {len(ann_files)} annotation files:')
for f, g in ann_files[:10]:
    print(f'  {f}')

if not ann_files:
    raise RuntimeError('Không tìm thấy annotation JSON cho các gesture đã download. '
                       'Check `ls -la /content/hagrid/annotations/` xem cấu trúc thật.')

# Build conversion map: image_id → list[bbox]
all_records = []   # (gesture, image_id, bboxes)
for ann_path, gesture in ann_files:
    with open(ann_path) as f:
        data = json.load(f)
    for image_id, info in data.items():
        bboxes = info.get('bboxes') or []
        if bboxes:
            all_records.append((gesture, image_id, bboxes))

print(f'\nTotal records: {len(all_records)} (with bboxes)')

# Shuffle + split
random.seed(config['seed'])
random.shuffle(all_records)
n_val = int(config['val_split'] * len(all_records))
splits = {'val': all_records[:n_val], 'train': all_records[n_val:]}
print(f'Train: {len(splits["train"])}, Val: {len(splits["val"])}')

# Convert + symlink images
missing = 0
for split_name, records in splits.items():
    for gesture, image_id, bboxes in tqdm(records, desc=split_name):
        # HaGRID image filename: {image_id}.jpg
        src_img = f'{img_root}/{gesture}/{image_id}.jpg'
        if not os.path.exists(src_img):
            missing += 1
            continue
        # symlink (fast, no copy)
        dst_img = f'{yolo_root}/images/{split_name}/{gesture}_{image_id}.jpg'
        if not os.path.exists(dst_img):
            os.symlink(src_img, dst_img)
        # write YOLO label: class 0 + cx cy w h
        dst_lbl = f'{yolo_root}/labels/{split_name}/{gesture}_{image_id}.txt'
        with open(dst_lbl, 'w') as f:
            for bx in bboxes:
                xtl, ytl, w, h = bx
                cx, cy = xtl + w/2, ytl + h/2
                f.write(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')

if missing:
    print(f'⚠️  {missing} ảnh không tìm thấy (image_id trong annotation nhưng file không có)')

# Generate dataset.yaml for Ultralytics
yaml_content = f"""path: {yolo_root}
train: images/train
val: images/val
names:
  0: hand
"""
yaml_path = f'{yolo_root}/dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)
print(f'\n✓ dataset.yaml written: {yaml_path}')
print(yaml_content)
!ls {yolo_root}/images/train | wc -l
!ls {yolo_root}/images/val | wc -l

## 5 · Train YOLOv8s với W&B logging

In [ ]:
import wandb
from ultralytics import YOLO, settings

wandb.login()
settings.update({'wandb': True})    # built-in Ultralytics → W&B integration

run = wandb.init(
    project=config['project_name'],
    name=config['run_name'],
    config=config,
    tags=['detector', 'yolov8s', 'hagrid'],
    notes=f'YOLOv8s fine-tuned on HaGRID gestures: {GESTURES_TO_DOWNLOAD}',
)

model = YOLO(config['model_variant'])

results = model.train(
    data=yaml_path,
    epochs=config['epochs'],
    batch=config['batch'],
    imgsz=config['imgsz'],
    patience=config['patience'],
    optimizer=config['optimizer'],
    lr0=config['lr0'],
    lrf=config['lrf'],
    momentum=config['momentum'],
    weight_decay=config['weight_decay'],
    warmup_epochs=config['warmup_epochs'],
    cos_lr=config['cos_lr'],
    mosaic=config['mosaic'],
    mixup=config['mixup'],
    amp=config['amp'],
    workers=config['workers'],
    seed=config['seed'],
    project='/content/runs',
    name=config['run_name'],
    exist_ok=True,
)

BEST_PT = f"/content/runs/{config['run_name']}/weights/best.pt"
LAST_PT = f"/content/runs/{config['run_name']}/weights/last.pt"
print(f'\n✓ Training done. Best: {BEST_PT}')

## 6 · Final validation + log metrics

In [ ]:
# Re-attach W&B run (Ultralytics built-in tự finish run sau train)
if wandb.run is None:
    try:
        wandb.init(project=run.project, id=run.id, resume='allow')
    except (NameError, AttributeError):
        wandb.init(project=config['project_name'], name=config['run_name'] + '-eval', config=config)

best = YOLO(BEST_PT)
metrics = best.val(data=yaml_path, imgsz=config['imgsz'], batch=config['batch'], split='val')

wandb.summary['final/mAP50']     = float(metrics.box.map50)
wandb.summary['final/mAP50_95']  = float(metrics.box.map)
wandb.summary['final/precision'] = float(metrics.box.mp)
wandb.summary['final/recall']    = float(metrics.box.mr)

print(f"mAP@0.5      = {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 = {metrics.box.map:.4f}")
print(f"Precision    = {metrics.box.mp:.4f}")
print(f"Recall       = {metrics.box.mr:.4f}")

## 7 · Sample predictions log lên W&B

In [ ]:
if wandb.run is None:
    try: wandb.init(project=run.project, id=run.id, resume='allow')
    except: pass

val_imgs = sorted(glob.glob(f'{yolo_root}/images/val/*.jpg'))
samples = random.sample(val_imgs, min(12, len(val_imgs)))
table = wandb.Table(columns=['image', 'num_detections', 'max_confidence'])
for p in samples:
    res = best.predict(source=p, conf=0.25, verbose=False)[0]
    plotted = res.plot()[..., ::-1]
    n = len(res.boxes); max_conf = float(res.boxes.conf.max()) if n > 0 else 0.0
    table.add_data(wandb.Image(plotted), n, max_conf)
wandb.log({'val/sample_predictions': table})
print(f'Logged {len(samples)} sample predictions.')

## 8 · Save artifact + copy về Drive + cleanup

In [ ]:
if wandb.run is None:
    try: wandb.init(project=run.project, id=run.id, resume='allow')
    except: pass

art = wandb.Artifact(
    name='yolov8s-hagrid-hand-detector', type='model',
    description=f'YOLOv8s fine-tuned on HaGRID {GESTURES_TO_DOWNLOAD}, mAP@0.5={metrics.box.map50:.4f}',
    metadata={
        'mAP50': float(metrics.box.map50),
        'mAP50_95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'gestures': GESTURES_TO_DOWNLOAD,
        'train_size': len(splits['train']),
        'val_size': len(splits['val']),
        'epochs': config['epochs'],
    },
)
art.add_file(BEST_PT, name='best.pt')
art.add_file(LAST_PT, name='last.pt')
wandb.log_artifact(art)

shutil.copy(BEST_PT, f'{SAVE_DIR}/yolov8s_hagrid_hand.pt')
print(f'✓ Saved:\n  Drive: {SAVE_DIR}/yolov8s_hagrid_hand.pt\n  W&B artifact: {art.name}')

wandb.finish()

# Optional cleanup — uncomment để giải phóng Colab disk
# shutil.rmtree(img_root, ignore_errors=True)
# print('Cleaned up extracted images.')

## Sau khi xong — chạy demo trên máy local

```bash
cd CV/Project/signlang
mkdir -p runs/detector/weights runs/classifier_resnet18
cp ~/Drive/MyDrive/signlang/yolov8s_hagrid_hand.pt runs/detector/weights/best.pt
cp ~/Drive/MyDrive/signlang/cnn_resnet18.pt runs/classifier_resnet18/best.pt
make demo
```

## Troubleshooting

- **`Unzip failed: invalid zip` ở cell 3** → URL của gesture đó nằm path khác. Sửa `GESTURE_URL_TEMPLATE` thành `GESTURE_URL_FALLBACK` rồi rerun.
- **`Disk full` lúc extract** → giảm số gestures trong `GESTURES_TO_DOWNLOAD`.
- **Train chậm hơn dự kiến** → `nvidia-smi` xem GPU util; nếu < 80% có thể do dataloader chậm — tăng `workers=8`.
- **Idle disconnect** → Colab Pro / chạy lại từ cell 5 (download/convert đã xong, chỉ retrain).